# Tutorial: from formula to centroids

This notebook walks through the enviPat pipeline for two compounds, glucose and
caffeine, plus the chlorinated pesticide chlorpyrifos to show a richer isotope pattern:

1. {func}`~envipat.formula.check_chemform`: validate formulas and compute
   monoisotopic masses,
2. {func}`~envipat.isopattern.isopattern`: isotope fine structure,
3. {func}`~envipat.envelope.envelope`: theoretical profile at a given resolution,
4. {func}`~envipat.vdetect.vdetect`: centroids of that profile,

and then the same in one call with {func}`~envipat.isowrap.isowrap` and a
resolution interpolated from an instrument curve with
{func}`~envipat.resolution.get_resolution`.

The notebook is executed on every documentation build, so all outputs below are
produced by the current version of the package.

In [ ]:
import matplotlib.pyplot as plt

import envipat as ep
import envipat.data as data

isotopes = data.isotopes
isotopes.head()

## 1. Check the formulas

`check_chemform` normalizes each formula (every element gets an explicit
count, bracketed groups are expanded) and flags invalid ones in the `warning`
column. The normalized `new_formula` is what the later steps use as keys.

In [ ]:
formulas = ["C6H12O6", "C8H10N4O2", "C9H11Cl3NO3PS", "C6H12Xx6"]
checked = ep.check_chemform(isotopes, formulas)
checked

The last formula contains an unknown element, so its `warning` is `True`.
Pipeline functions expect valid formulas, so keep only the valid rows:

In [ ]:
checked = checked[~checked["warning"]].reset_index(drop=True)
chlorpyrifos = checked["new_formula"][2]  # "C9H11Cl3N1O3P1S1"
checked

## 2. Isotope fine structure

`isopattern` returns a `dict` keyed by formula. Each value is a DataFrame with
the peak `m/z`, its `abundance` relative to the most intense peak (default
`rel_to=0`) and the isotope counts of every peak. Peaks below `threshold`
(in percent) are pruned. `charge=1` computes m/z of the singly charged
ion (electron mass subtracted).

In [ ]:
pattern = ep.isopattern(
    isotopes,
    checked["new_formula"],
    threshold=0.1,
    charge=1,
    algo=2,
    verbose=False,
)
pattern[chlorpyrifos].head(10)

## 3. Theoretical profile

`envelope` convolutes each stick pattern with a Gaussian (or
`env="CauchyLorentz"`) peak shape. The peak width follows from the resolving
power, `resolution`: FWHM = m/z / resolution. With `dmz="get"` the m/z step is
derived from the resolution (`frac` of the FWHM).

In [ ]:
profiles = ep.envelope(pattern, dmz="get", frac=0.1, resolution=60_000, verbose=False)
profiles[chlorpyrifos].head()

## 4. Centroids

`vdetect` with `detect="centroid"` returns one m/z/abundance pair per resolved
profile peak. At this resolution, fine-structure peaks that lie closer together
than the peak width merge into a single centroid.

In [ ]:
centroids = ep.vdetect(profiles, detect="centroid", verbose=False)
centroids[chlorpyrifos]

In [ ]:
def plot_compound(formula, ax):
    sticks = pattern[formula]
    profile = profiles[formula]
    cents = centroids[formula]
    ax.plot(profile["m/z"], profile["abundance"], lw=1, label="profile")
    ax.vlines(sticks["m/z"], 0, sticks["abundance"], color="grey", lw=1, label="fine structure")
    ax.plot(cents["m/z"], cents["abundance"], "o", ms=4, label="centroids")
    ax.set(title=formula, xlabel="m/z", ylabel="relative abundance")


fig, ax = plt.subplots(figsize=(8, 4))
plot_compound(chlorpyrifos, ax)
ax.legend()
fig.tight_layout()

Zooming in on the M+1 cluster of caffeine: at a resolving power of 60 000
(FWHM about 0.003 at m/z 195) the 15N and 13C isotopologues are resolved, but
the 2H isotopologue, 0.003 above the 13C one, merges into the 13C peak.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_compound("C8H10N4O2", ax)
ax.set_xlim(195.07, 195.095)
ax.set_ylim(0, 10)
ax.ticklabel_format(axis="x", useOffset=False)
ax.legend()
fig.tight_layout()

## 5. Everything in one call

`isowrap` runs steps 2-4. Instead of one fixed `resolution`, real instruments
have a resolving power that depends on m/z. The bundled
{data}`~envipat.data.resolution_list` holds calibration curves for common
instruments; `get_resolution` interpolates one at each formula's mass.

In [ ]:
resmass = data.resolution_list["Elite_R240000@400"]
resolution = ep.get_resolution(checked, resmass)
resolution

In [ ]:
result = ep.isowrap(
    isotopes,
    checked,
    resolution=resolution,
    threshold=0.1,
    charge=1,
    detect="centroid",
    verbose=False,
)
result["C8H10N4O2"]

This curve gives a resolving power of about 350 000 at caffeine's mass, so
the 2H isotopologue at m/z 195.086 now gets its own centroid, next to the
15N and 13C ones.

## Results that failed

Every pipeline function maps a formula whose calculation failed to the string
`"error"` instead of a DataFrame, as the R package does. Check before use:

In [ ]:
for formula, peaks in result.items():
    if isinstance(peaks, str):
        print(f"{formula}: calculation failed")
    else:
        base_mz = peaks["m/z"].iloc[peaks["abundance"].argmax()]
        print(f"{formula}: {len(peaks)} centroids, base peak at m/z {base_mz:.4f}")